In [1]:
import os
try:
    from yaml import load, CLoader as Loader
except:
    from yaml import load, Loader

data = load(open('env.yaml'), Loader=Loader)
os.environ.update({key: str(val) for key, val in data.items()})

from hemlock import push_app_context

app = push_app_context()

In [2]:
from hemlock import Check, Compile as C

check = Check(
    '<p>Select the correct answer.</p>',
    ['Correct', 'Incorrect', 'Also incorrect'],
    compile=C.shuffle()
)
check.compile

[<Compile 1>]

In [3]:
check._compile()
[choice.label for choice in check.choices]

['Incorrect', 'Correct', 'Also incorrect']

In [4]:
from hemlock import Input, Label, Page, Range, Select
from hemlock.tools import join

@C.register
def confirm(confirm_label, demographics_page):
    # get the participant's data from the demographics page
    demographics = [q.data for q in demographics_page.questions]
    # re-format the race demographic data
    race = demographics_page.questions[2]
    race = join('and', *(key for key in race.data if race.data[key]))
    demographics[2] = race
    # set the label based on the participant's demographics data
    confirm_label.label = '''
    <p>Confirm the following information:</p>
    <ul>
        <li>Date of birth: {}</li>
        <li>Gender: {}</li>
        <li>Race/Ethnicity: {}</li>
        <li>Marital status: {}</li>
        <li>Subjective socio-economic status: {}</li>
    </ul>
    <p>To correct this information, click '<<'.</p>
    '''.format(*demographics)

demographics_page = Page(
    Input(data='10/26/1992'),
    Check(data='Male'),
    Check(data={'White': 1, 'Black': 1}),
    Select(data='Never married'),
    Range(data='5')
)
confirm_page = Page(
    Label(compile=C.confirm(demographics_page)), 
    back=True
)
confirm_page._compile().preview()

<Page 2>

In [5]:
[os.remove(tmpfile) for tmpfile in app.tmpfiles]

[None]